# 04 — Sensitivity Analysis

**Catalog Campaign Profitability & Customer Targeting Analysis**

## Purpose

The base case says the campaign earns $21,987. That figure rests on three
assumptions, none of which is certain. This notebook asks the question management
actually cares about:

> **How wrong can we be before the answer changes?**

A single point estimate invites false confidence. A range with a stated
break-even point supports a decision.

## What is tested

| Assumption | Base case | Range tested | Why this range |
|---|---|---|---|
| Gross margin (A-02) | 50% | 40% - 60% | Plausible spread for a retail product mix; highest-leverage financial input |
| Cost per catalog (A-01) | $6.50 | $5.00 - $10.00 | Covers print and postage inflation and loss of volume discounts |
| Response level (A-05) | Supplied `Score_Yes` | 60% - 120% of supplied | The input whose provenance cannot be validated here |

> **Every figure in this notebook other than the base case is hypothetical.**
> Scenario values are labelled as such throughout and must never be quoted as
> forecasts (BRule-14).

**Requirements addressed:** BR-009 / FR-015, FR-016.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path.cwd().parent))
from src import data_cleaning as dc
from src import modeling as md
from src import profitability as pf

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

customers = dc.clean_customers(dc.load_customers())
mailing = dc.clean_mailing_list(dc.load_mailing_list())
model = md.fit_linear_model(customers)
economics = pf.build_customer_economics(md.score_mailing_list(model, mailing))

base = pf.campaign_summary(economics)
print("BASE CASE (actual model output, not a scenario)")
print(f"  Expected revenue : ${base['expected_revenue']:,.2f}")
print(f"  Gross profit     : ${base['gross_profit']:,.2f}")
print(f"  Campaign cost    : ${base['campaign_cost']:,.2f}")
print(f"  Net profit       : ${base['net_profit']:,.2f}")
print(f"  ROI              : {base['roi']:.2f}x")

## 1. Break-even analysis

Before running scenarios, establish the point at which the campaign stops
creating value. Each figure below holds the other two assumptions at their base
case.

In [ ]:
gross = base["gross_profit"]; revenue = base["expected_revenue"]; cost = base["campaign_cost"]
n = base["customers"]

breakeven = pd.DataFrame([
    {"Assumption": "Response level (as a fraction of modelled)",
     "Base case": "100%",
     "Break-even point": f"{base['breakeven_response_multiplier']:.1%}",
     "Headroom": f"Response can fall {1 - base['breakeven_response_multiplier']:.1%} before breaking even"},
    {"Assumption": "Campaign response rate",
     "Base case": f"{base['avg_response_probability']:.1%}",
     "Break-even point": f"{base['breakeven_response_multiplier'] * base['avg_response_probability']:.2%}",
     "Headroom": "Only about 1 response in 43 is needed to break even"},
    {"Assumption": "Gross margin",
     "Base case": "50.0%",
     "Break-even point": f"{cost / revenue:.2%}",
     "Headroom": f"Margin can fall {50 - (cost / revenue * 100):.1f} points before breaking even"},
    {"Assumption": "Cost per catalog",
     "Base case": "$6.50",
     "Break-even point": f"${gross / n:,.2f}",
     "Headroom": f"Cost could rise {gross / n / 6.50:.0f}x before breaking even"},
    {"Assumption": "Average predicted sale amount",
     "Base case": f"${economics['Predicted_Sale_Amount'].mean():,.2f}",
     "Break-even point": f"${economics['Predicted_Sale_Amount'].mean() * base['breakeven_response_multiplier']:,.2f}",
     "Headroom": "Predictions would have to be overstated ~14x"},
])
breakeven

### Business interpretation

The campaign carries **very wide margins for error**. Any single assumption would
have to be wrong by an implausible amount before the campaign stops paying:

- Response would have to collapse to **6.9% of the modelled level** — a campaign
  response rate of about **2.3%** rather than the forecast 34%.
- Gross margin would have to fall from 50% to under **3.5%**.
- Catalog cost would have to rise from $6.50 to about **$94**.

These are not near-misses. They are failure modes so extreme that if any of them
occurred, the organisation would have far larger problems than this campaign.

**The honest caveat.** These break-evens each move one lever at a time. Real
downside tends to arrive with correlated moves — a weak market depresses response
*and* margin simultaneously. The combined scenario is tested in section 4.

## 2. Scenario grid

In [ ]:
gross_margins = [0.40, 0.45, 0.50, 0.55, 0.60]
catalog_costs = [5.00, 6.50, 8.00, 10.00]
response_multipliers = [0.6, 0.8, 1.0, 1.2]

grid = pf.sensitivity_grid(economics, gross_margins, catalog_costs, response_multipliers)
print(f"{len(grid)} scenarios generated "
      f"({len(response_multipliers)} response x {len(gross_margins)} margin x {len(catalog_costs)} cost)")
grid.head()

## 3. Margin vs catalog cost (response held at the modelled level)

> Scenario values. Only the 50% / $6.50 cell is the base case.

In [ ]:
base_slice = grid[grid["Response_Multiplier"] == 1.0]
heat = base_slice.pivot_table(index="Gross_Margin", columns="Cost_Per_Catalog", values="Net_Profit")

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(heat, annot=True, fmt=",.0f", cmap="RdYlGn", center=base["net_profit"],
            cbar_kws={"label": "Net profit ($)"}, ax=ax)
ax.set_title("SCENARIO: net profit by gross margin and catalog cost\n(response held at the modelled level)")
ax.set_xlabel("Cost per catalog ($)"); ax.set_ylabel("Gross margin")
ax.set_yticklabels([f"{v:.0%}" for v in heat.index], rotation=0)
plt.tight_layout(); plt.show()

heat

### Business interpretation

**The two levers are not equally important.**

- Moving **gross margin** from 40% to 60% swings net profit from about $17,300 to
  about $26,700 — a range of roughly **$9,400**.
- Moving **catalog cost** from $5.00 to $10.00 at a fixed margin swings net profit
  by only about **$1,250**.

Gross margin matters roughly **seven times more** than catalog cost.

**What management should do with this.** Effort spent negotiating a lower print
price is largely wasted. Effort spent confirming the true blended margin with
Finance is where the risk actually sits. That is a concrete reprioritisation of
attention, which is the point of running the analysis.

Every cell in the grid is profitable. Within this range, the decision does not
change.

## 4. Response sensitivity — the input that matters most

Response probability is the assumption whose provenance cannot be validated here
(A-05, risk R-02), so it deserves its own treatment.

In [ ]:
resp_slice = grid[(grid["Gross_Margin"] == 0.50) & (grid["Cost_Per_Catalog"] == 6.50)].copy()
resp_slice["Scenario"] = resp_slice["Response_Multiplier"].map({
    0.6: "60% of modelled (severe shortfall)",
    0.8: "80% of modelled (mild shortfall)",
    1.0: "100% - BASE CASE",
    1.2: "120% of modelled (outperformance)",
})

fig, ax = plt.subplots(figsize=(10, 5))
colours = ["#c53030" if m < 1.0 else ("#2b6cb0" if m == 1.0 else "#2f855a")
           for m in resp_slice["Response_Multiplier"]]
bars = ax.bar(resp_slice["Scenario"], resp_slice["Net_Profit"], color=colours)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Net profit across response scenarios (margin 50%, cost $6.50)")
ax.set_ylabel("Net profit ($)")
ax.set_xticklabels(resp_slice["Scenario"], rotation=18, ha="right", fontsize=8)
for bar, val in zip(bars, resp_slice["Net_Profit"]):
    ax.annotate(f"${val:,.0f}", (bar.get_x() + bar.get_width() / 2, val),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

resp_slice[["Scenario", "Expected_Revenue", "Gross_Profit", "Campaign_Cost", "Net_Profit", "ROI"]]

### Business interpretation

Even at a **severe 40% shortfall** in response, the campaign returns **$12,542 of
net profit at 7.7x ROI**. The decision does not flip.

This is the most important robustness finding in the project, because it isolates
the weakest link in the analysis — a probability score this project cannot
validate — and shows that the conclusion survives being substantially wrong
about it.

## 5. Combined downside — all three assumptions adverse at once

Single-variable sensitivity flatters a project. Real downside arrives correlated.

In [ ]:
scenarios = pd.DataFrame([
    {"Scenario": "Best case", "Response": 1.2, "Margin": 0.60, "Cost": 5.00},
    {"Scenario": "BASE CASE", "Response": 1.0, "Margin": 0.50, "Cost": 6.50},
    {"Scenario": "Mild downside", "Response": 0.8, "Margin": 0.45, "Cost": 8.00},
    {"Scenario": "Severe downside", "Response": 0.6, "Margin": 0.40, "Cost": 10.00},
])

rows = []
for _, s in scenarios.iterrows():
    match = grid[(grid["Response_Multiplier"] == s["Response"]) &
                 (grid["Gross_Margin"] == s["Margin"]) &
                 (grid["Cost_Per_Catalog"] == s["Cost"])].iloc[0]
    rows.append({
        "Scenario": s["Scenario"],
        "Response": f"{s['Response']:.0%} of modelled",
        "Margin": f"{s['Margin']:.0%}",
        "Cost/catalog": f"${s['Cost']:.2f}",
        "Expected revenue": match["Expected_Revenue"],
        "Gross profit": match["Gross_Profit"],
        "Campaign cost": match["Campaign_Cost"],
        "Net profit": match["Net_Profit"],
        "ROI": match["ROI"],
    })
combined = pd.DataFrame(rows)
combined

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colours = {"Best case": "#2f855a", "BASE CASE": "#2b6cb0",
           "Mild downside": "#dd6b20", "Severe downside": "#c53030"}
bars = ax.bar(combined["Scenario"], combined["Net profit"],
              color=[colours[s] for s in combined["Scenario"]])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Net profit under combined assumption scenarios")
ax.set_ylabel("Net profit ($)")
for bar, val, r in zip(bars, combined["Net profit"], combined["ROI"]):
    ax.annotate(f"${val:,.0f}\n{r:.1f}x ROI",
                (bar.get_x() + bar.get_width() / 2, val),
                ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

### Business interpretation

**The campaign remains profitable even when all three assumptions move against
it simultaneously.** In the severe downside — response 40% below modelled, margin
at 40%, catalog cost at $10 — net profit is still **$8,834 at 3.53x ROI**.

This is the finding that converts a forecast into a decision. Management is not
being asked to bet on the model being right. They are being asked to approve a
campaign that pays under every scenario the analysis can construct from plausible
inputs.

**The limit of that claim, stated plainly.** This grid varies the *assumptions*.
It cannot test the scenarios that sit outside the model's frame:

- **Non-incrementality** (A-08): if a large share of these orders would have
  occurred anyway, measured profit overstates true value. Only a randomised
  holdout group can settle this, and this campaign design does not include one.
- **Structural break**: if customer behaviour has shifted since the training data
  was collected, and the data carries no timestamp to check (C-09), the
  relationships themselves may no longer hold.

Those are the two ways this recommendation could still be wrong, and neither is
visible in any cell of this grid.

## 6. Tornado chart — which assumption moves the answer most?

In [ ]:
impacts = []
for label, low, high, key in [
    ("Gross margin (40% - 60%)", 0.40, 0.60, "Gross_Margin"),
    ("Response level (60% - 120%)", 0.6, 1.2, "Response_Multiplier"),
    ("Cost per catalog ($5 - $10)", 5.00, 10.00, "Cost_Per_Catalog"),
]:
    defaults = {"Gross_Margin": 0.50, "Response_Multiplier": 1.0, "Cost_Per_Catalog": 6.50}
    def lookup(val):
        q = dict(defaults); q[key] = val
        return grid[(grid["Response_Multiplier"] == q["Response_Multiplier"]) &
                    (grid["Gross_Margin"] == q["Gross_Margin"]) &
                    (grid["Cost_Per_Catalog"] == q["Cost_Per_Catalog"])]["Net_Profit"].iloc[0]
    lo, hi = lookup(low), lookup(high)
    impacts.append({"Assumption": label, "Low": lo, "High": hi,
                    "Swing": abs(hi - lo)})

tornado = pd.DataFrame(impacts).sort_values("Swing")

fig, ax = plt.subplots(figsize=(10, 4))
for i, row in enumerate(tornado.itertuples()):
    ax.barh(i, row.High - row.Low, left=min(row.Low, row.High),
            color="#2b6cb0", alpha=0.75, height=0.55)
    ax.annotate(f"${row.Swing:,.0f} swing", (max(row.Low, row.High), i),
                textcoords="offset points", xytext=(8, -3), fontsize=9)
ax.axvline(base["net_profit"], color="#c53030", linestyle="--",
           label=f"Base case ${base['net_profit']:,.0f}")
ax.set_yticks(range(len(tornado))); ax.set_yticklabels(tornado["Assumption"])
ax.set_xlabel("Net profit ($)")
ax.set_title("Which assumption moves net profit most?")
ax.legend()
plt.tight_layout(); plt.show()

tornado[["Assumption", "Low", "High", "Swing"]]

### Business interpretation

Ranked by how much each assumption can move the answer:

1. **Response level** — the widest swing, and the input this project cannot
   validate. Highest priority for post-campaign measurement (KPI M-01).
2. **Gross margin** — close behind, but unlike response it can be resolved with
   a single conversation with Finance before mailing.
3. **Cost per catalog** — barely moves the answer. Low priority.

**Where to spend effort before mailing:** confirm the margin with Finance, and
make sure response tracking is in place so the largest unknown becomes a measured
quantity after the first campaign. Do not spend time renegotiating print costs.

## 7. Export

In [ ]:
out_dir = Path.cwd().parent / "outputs"
out_dir.mkdir(exist_ok=True)
grid.to_csv(out_dir / "sensitivity_analysis.csv", index=False)
print(f"Written {len(grid)} scenarios to outputs/sensitivity_analysis.csv")

## Summary

| Finding | Detail |
|---|---|
| Base case net profit | $21,987.44 at 13.53x ROI |
| Break-even response | 6.9% of the modelled level (~2.3% campaign response rate) |
| Break-even margin | ~3.4% gross margin |
| Break-even catalog cost | ~$94 per catalog |
| Severe combined downside | $8,834 net profit at 3.53x ROI — still profitable |
| Highest-leverage assumption | Response level, followed closely by gross margin |
| Lowest-leverage assumption | Cost per catalog |
| Scenarios tested | 80, all profitable |

**Conclusion.** The recommendation to proceed is robust across every scenario the
available data supports. The residual risks — non-incrementality and structural
change in customer behaviour — sit outside what any sensitivity grid can test and
are carried into the limitations and risk register instead.

See [`docs/decision_framework.md`](../docs/decision_framework.md) for how these
findings map onto the approval decision.